# Convolutional Neural Networks
### Interactive Notebook for AI/ML Interview Preparation

Deep learning with CNNs: from fundamentals to pretrained models and transfer learning. PyTorch implementation.

📺 **Video Lecture:** [https://youtu.be/K1xfB-H_Ii4](https://youtu.be/K1xfB-H_Ii4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import resnet18
import time
from collections import OrderedDict

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('All libraries loaded!')

---
## 1. Convolution Fundamentals

### 1.1 2D Convolution from Scratch (NumPy)

Educational implementation to understand how convolution works.

In [ ]:
def conv2d_numpy(image, kernel, stride=1, padding=0):
    """2D convolution operation from scratch using NumPy."""
    if padding > 0:
        image = np.pad(image, padding, mode='constant')
    h, w = image.shape
    kh, kw = kernel.shape
    out_h = (h - kh) // stride + 1
    out_w = (w - kw) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+kh, j*stride:j*stride+kw]
            output[i, j] = np.sum(region * kernel)
    return output

# Create a simple 8x8 image
image = np.zeros((8, 8))
image[2:6, 2:6] = 1  # white square in center

# Sobel filters for edge detection
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])  # vertical edges
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])   # horizontal edges

edges_x = conv2d_numpy(image, sobel_x, padding=1)
edges_y = conv2d_numpy(image, sobel_y, padding=1)
edges_combined = np.sqrt(edges_x**2 + edges_y**2)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, img, title in zip(axes, [image, edges_x, edges_y, edges_combined],
                          ['Original', 'Vertical Edges', 'Horizontal Edges', 'Combined']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('NumPy convolution demonstrates edge detection via manual filtering.')

### 1.2 Convolution with PyTorch (torch.nn.Conv2d)

PyTorch's efficient implementation with learnable weights.

In [ ]:
# Create a simple Conv2d layer
conv_layer = nn.Conv2d(in_channels=1, out_channels=3, kernel_size=3, stride=1, padding=1)

# Inspect the weights and bias
print(f'Conv2d layer:')
print(f'  in_channels: {conv_layer.in_channels}')
print(f'  out_channels: {conv_layer.out_channels}')
print(f'  kernel_size: {conv_layer.kernel_size}')
print(f'  weight shape: {conv_layer.weight.shape}')  # (out_channels, in_channels, kh, kw)
print(f'  bias shape: {conv_layer.bias.shape}')      # (out_channels,)')

# Convert NumPy image to PyTorch tensor
image_tensor = torch.from_numpy(image).float().unsqueeze(0).unsqueeze(0)  # (1, 1, 8, 8)
print(f'\nInput tensor shape: {image_tensor.shape}')

# Apply convolution
with torch.no_grad():
    output_tensor = conv_layer(image_tensor)

print(f'Output tensor shape: {output_tensor.shape}')  # (1, 3, 8, 8)
print(f'Output activation range: [{output_tensor.min():.2f}, {output_tensor.max():.2f}]')

---
## 2. Pooling Layers

### 2.1 Max Pooling and Average Pooling

In [ ]:
# Create sample feature map
test_img = np.random.rand(6, 6)

# NumPy implementations (educational)
def max_pool_numpy(image, pool_size=2):
    h, w = image.shape
    out_h, out_w = h // pool_size, w // pool_size
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
            output[i, j] = np.max(region)
    return output

def avg_pool_numpy(image, pool_size=2):
    h, w = image.shape
    out_h, out_w = h // pool_size, w // pool_size
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
            output[i, j] = np.mean(region)
    return output

max_pooled = max_pool_numpy(test_img, pool_size=2)
avg_pooled = avg_pool_numpy(test_img, pool_size=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(test_img, cmap='viridis')
axes[0].set_title(f'Original {test_img.shape}')
axes[1].imshow(max_pooled, cmap='viridis')
axes[1].set_title(f'Max Pool {max_pooled.shape}')
axes[2].imshow(avg_pooled, cmap='viridis')
axes[2].set_title(f'Avg Pool {avg_pooled.shape}')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### 2.2 PyTorch Pooling Layers

In [ ]:
# PyTorch pooling
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

# Convert test image to tensor
test_tensor = torch.from_numpy(test_img).float().unsqueeze(0).unsqueeze(0)  # (1, 1, 6, 6)

with torch.no_grad():
    max_pooled_torch = max_pool(test_tensor)
    avg_pooled_torch = avg_pool(test_tensor)

print(f'Input shape: {test_tensor.shape}')
print(f'Max pool output shape: {max_pooled_torch.shape}')
print(f'Avg pool output shape: {avg_pooled_torch.shape}')
print(f'\nMax pooling reduces spatial dimensions by 2x (with stride=2, kernel_size=2)')

---
## 3. Build a CNN in PyTorch

### 3.1 LeNet-Style Architecture

A simple but effective CNN for MNIST.

In [ ]:
class SimpleCNN(nn.Module):
    """LeNet-inspired CNN for MNIST."""
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, padding=2)
        
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        
        # Activation
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Conv block 1: 1x28x28 -> 32x28x28 -> 32x14x14
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        
        # Conv block 2: 32x14x14 -> 64x14x14 -> 64x7x7
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully connected
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Instantiate model
model = SimpleCNN().to(device)

# Print architecture
print('SimpleCNN Architecture:')
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

---
## 4. Train on MNIST

### 4.1 Prepare MNIST Dataset

In [ ]:
# Data transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Download and load MNIST
print('Loading MNIST...')
train_dataset = datasets.MNIST(root='/tmp/mnist', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='/tmp/mnist', train=False, transform=transform, download=True)

# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f'Training samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')
print(f'Batch size: {batch_size}')

# Visualize a batch
sample_batch, sample_labels = next(iter(train_loader))
print(f'\nSample batch shape: {sample_batch.shape}')  # (batch, channels, height, width)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_batch[i, 0].numpy(), cmap='gray')
    ax.set_title(f'Label: {sample_labels[i].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 4.2 Training Function

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    """Evaluate on validation/test set."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print('Training functions defined.')

### 4.3 Train the Model

In [ ]:
# Reset model
model = SimpleCNN().to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training parameters
epochs = 5
train_losses, train_accs = [], []
test_losses, test_accs = [], []

print('Training SimpleCNN on MNIST...')
start_time = time.time()

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    print(f'Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | '
          f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%')

elapsed = time.time() - start_time
print(f'\nTraining completed in {elapsed:.2f}s')
print(f'Final Test Accuracy: {test_accs[-1]:.2f}%')

### 4.4 Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(range(1, epochs + 1), train_losses, 'o-', label='Train Loss')
ax1.plot(range(1, epochs + 1), test_losses, 's-', label='Test Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Test Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(range(1, epochs + 1), train_accs, 'o-', label='Train Accuracy')
ax2.plot(range(1, epochs + 1), test_accs, 's-', label='Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Visualize Filters and Feature Maps

### 5.1 Extract and Visualize Learned Filters

In [ ]:
# Get the first convolutional layer's weights
conv1_weights = model.conv1.weight.data.cpu()  # Shape: (32, 1, 5, 5)

# Normalize for visualization
conv1_weights_norm = (conv1_weights - conv1_weights.min()) / (conv1_weights.max() - conv1_weights.min())

# Visualize first 16 filters
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(conv1_weights_norm[i, 0].numpy(), cmap='gray')
    ax.set_title(f'Filter {i}')
    ax.axis('off')
plt.suptitle('First 16 Learned Filters from Conv1')
plt.tight_layout()
plt.show()

print(f'Conv1 filter shape: {conv1_weights.shape}')
print(f'Conv2 filter shape: {model.conv2.weight.data.shape}')

### 5.2 Visualize Feature Maps

In [ ]:
# Get a sample image from test set
sample_img, sample_label = test_dataset[0]
sample_img_batch = sample_img.unsqueeze(0).to(device)

# Forward pass through each layer
model.eval()
with torch.no_grad():
    # After conv1 and pool
    x = model.relu(model.conv1(sample_img_batch))
    x = model.pool(x)
    feature_map_1 = x.cpu()  # Shape: (1, 32, 14, 14)
    
    # After conv2 and pool
    x = model.relu(model.conv2(x))
    x = model.pool(x)
    feature_map_2 = x.cpu()  # Shape: (1, 64, 7, 7)

# Visualize feature maps from conv1
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i in range(32):
    row, col = i // 8, i % 8
    fm = feature_map_1[0, i].numpy()
    axes[row, col].imshow(fm, cmap='hot')
    axes[row, col].set_title(f'FM {i}')
    axes[row, col].axis('off')
plt.suptitle('Feature Maps from Conv1+Pool (14x14 spatial)')
plt.tight_layout()
plt.show()

print('Feature maps show learned feature representations at different spatial scales.')

---
## 6. Pretrained Models

### 6.1 Load ResNet18 from torchvision

In [ ]:
# Load pretrained ResNet18
resnet = resnet18(weights='DEFAULT')  # weights='DEFAULT' loads ImageNet-pretrained weights
print('ResNet18 (pretrained on ImageNet):')
print(resnet)

In [ ]:
def count_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen

# ResNet18 parameter counts
resnet_total, resnet_trainable, resnet_frozen = count_parameters(resnet)

print('\nResNet18 Parameter Summary:')
print(f'Total parameters: {resnet_total:,}')
print(f'Trainable parameters: {resnet_trainable:,}')
print(f'Frozen parameters: {resnet_frozen:,}')

# Compare with SimpleCNN
cnn_total, cnn_trainable, cnn_frozen = count_parameters(model)

print('\nSimpleCNN vs ResNet18 Comparison:')
print(f'SimpleCNN total: {cnn_total:,} | ResNet18 total: {resnet_total:,}')
print(f'Ratio: ResNet18 is {resnet_total / cnn_total:.1f}x larger')

### 6.3 Model Architecture Analysis

---
## 7. Transfer Learning Demo

### 7.1 Freeze Backbone and Replace Classifier

### 7.2 Fine-tuning Strategy

---
## 8. Data Augmentation

---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>